# 概要

動物の鳴き声(牛、熊)のデータセットを用いて<br>
3層のSAMACTが音声の特徴を学習し、推論するサンプルシナリオ。<br>
牛の音声は時間が長いので、学習用データセットのファイル数を熊の半分にした。
    
1. データの準備(前処理含む)
2. ニューラルネットワークの構築<br>
  2-1. エンコーダの設定<br>
  2-2. SAMレイヤーの設定<br>
  2-3. デコーダの設定<br>
  2-4. ニューラルネットワークの生成
3. 学習<br>
  3-1. 学習プロパティの設定<br>
  3-2. 学習の実施<br>
  3-3. 学習結果の保存
4. 推論<br>
  4-1. テスト用データセットをまとめて推論<br>
  4-2. 1データを推論


# 1. データの準備(前処理含む)
今回、前処理にメルスペクトログラムを利用。

In [1]:
from pathlib import Path

import numpy as np
import librosa

# 前処理とmainで共通して利用する変数
N_N1 = 100 # 入力層の数

# 前処理で正規化するときに使う範囲。(全次元共通)
# 個別に情報量最大になる範囲を設定すると、より良い結果を得られる。
MIN_MEL = -50
MAX_MEL = 0

def MakePreProcessedDataset(basePath:Path)->tuple[np.ndarray, np.ndarray]:
    """
    data, labelを返す。
    この返り値をSAMACTに渡せばよい。
    """

    dataset = []
    labels = []

    # 探索順の再現性のためにsortedを使用(iでラベルを貼るため)
    # 各種wavファイルのmelspectrogramを計算し、正常/異常のラベルを貼って、行列としてまとめる。
    for i, labelDirPath in enumerate(sorted(basePath.glob("*/"))):
        datasetOnLabel = []

        # ファイルの順番の再現性のためにsortedを使用
        for wav in sorted(labelDirPath.glob("*.wav")):
            y, sr = librosa.load(path=wav, sr=None)
            melSpec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_N1,
                                                     n_fft=2048, hop_length=256)
            melSpecDb = librosa.power_to_db(melSpec, ref=1.0,
                                            amin=1e-10, top_db=80.0)
            # librosaの返り値は想定する行列と、軸が逆なので転置してスケーリング
            normMelSpecDb = (melSpecDb.T - MIN_MEL) / (MAX_MEL - MIN_MEL)

            datasetOnLabel.append(normMelSpecDb)

        naDatasetOnLabel = np.vstack(datasetOnLabel)
        dataset.append(naDatasetOnLabel)

        labels.extend([i]*len(naDatasetOnLabel))

    # SAMACTはオンライン学習するため、ラベルの偏りが悪影響を与える。
    # シャッフルすることで対策する。
    naDataset = np.vstack(dataset)
    naLabels = np.array(labels)
    np.random.seed(0)
    indice = np.arange(len(naDataset))
    np.random.shuffle(indice)
    datasetShuffled = naDataset[indice, :]
    labelShuffled = naLabels[indice]

    return datasetShuffled, labelShuffled

In [2]:
trainPath = Path("data/AnimalCry/train")
testPath = Path("data/AnimalCry/test")

list(trainPath.glob('*/*.wav')), list(testPath.glob('*/*.wav'))

([PosixPath('data/AnimalCry/train/0_normal/cow_1.wav'),
  PosixPath('data/AnimalCry/train/0_normal/cow_2.wav'),
  PosixPath('data/AnimalCry/train/1_anomaly/bear_1.wav'),
  PosixPath('data/AnimalCry/train/1_anomaly/bear_2.wav'),
  PosixPath('data/AnimalCry/train/1_anomaly/bear_3.wav'),
  PosixPath('data/AnimalCry/train/1_anomaly/bear_4.wav')],
 [PosixPath('data/AnimalCry/test/0_normal/cow_3.wav'),
  PosixPath('data/AnimalCry/test/1_anomaly/bear_5.wav')])

In [3]:
trainData, trainLabel = MakePreProcessedDataset(trainPath)
testData, testLabel = MakePreProcessedDataset(testPath)
trainData.shape, trainLabel.shape

((1356, 100), (1356,))

## 数値について
特徴量の数値は、フレームワーク内部で下記のように丸められる。
- 0未満の数値 → 0
- 1より大きい数値 → 1

そのため、本サンプルの正規化部分ではclipを省略している。

In [4]:
# dataをprintすると、負の数も見られるが上記の仕様によりフレームワーク内部では0とみなされる
trainData, trainLabel

(array([[ 0.35583436,  0.55025184,  0.5035559 , ..., -0.10606659,
         -0.10606659, -0.10606659],
        [ 0.4418325 ,  0.43599743,  0.52515626, ..., -0.03677956,
         -0.03677956, -0.03677956],
        [ 0.6662288 ,  0.5735254 ,  0.52227575, ..., -0.10374077,
         -0.10374077, -0.10374077],
        ...,
        [ 0.5977253 ,  0.4771198 ,  0.5244991 , ..., -0.10525032,
         -0.10525032, -0.10525032],
        [ 0.42890778,  0.4706791 ,  0.50138056, ..., -0.03677956,
         -0.03677956, -0.03677956],
        [ 0.3223564 ,  0.35213462,  0.5420098 , ..., -0.03677956,
         -0.03677956, -0.03677956]], dtype=float32),
 array([1, 0, 1, ..., 1, 0, 0]))

## ラベル
ラベルは、0から順番にインクリメントされるint型を想定しているので<br>
用意したデータがそうでない場合は、上記のフォーマットに合わせる。今回は修正不要。

In [5]:
trainLabel.shape, trainLabel.dtype

((1356,), dtype('int64'))

# 2. ニューラルネットワークの構築
SAMACTのフレームワークの各要素をインスタンスして、ニューラルネットワークを構築する。

本サンプルシナリオでは、3層(100-200-2)のニューラルネットワークを構築する。

In [6]:
from samact import *

## 2-1. エンコーダの設定
エンコーダとして、RateEncodeLayerを指定。<br>
パラメータnUnitsは入力する特徴量の数と一致させる。

In [7]:
inputLayer = RateEncodeLayer(N_N1)
inputLayer

RateEncode(nUnits=100)

## 2-2. SAMレイヤーの設定(隠れ層)

隠れ層として、SAMLayerを指定。

In [8]:
hiddenLayer = SAMLayer(200, LayerProperty(a=3, p=0.75), Step(), Step())
hiddenLayer

SAMLayer(nUnits=200, gradAct=Step(a=0, b=6, g=18), teacherAct=Step(a=0, b=6, g=18))

## 2-2. SAMレイヤーの設定(出力層)
出力層として、SAMLayerを指定。<br>
パラメータnUnitsは分類するカテゴリ数と一致させる。

In [9]:
outputLayer = SAMLayer(2, LayerProperty(a=3, p=0.75), Step(), Step())
outputLayer

SAMLayer(nUnits=2, gradAct=Step(a=0, b=6, g=18), teacherAct=Step(a=0, b=6, g=18))

## 2-3. デコーダの設定

デコーダとして、MajorityDecodeLayerを定義。<br>
分類問題に対しては、基本的にこのデコーダを指定する。

In [10]:
decoder = MajorityDecodeLayer()
decoder

MajorityDecode

## 2-4. ネットワークの設定
Sequentialモデルとして、SAMACTのネットワークを定義。

In [11]:
model = Sequential(inputLayer, decoder, [hiddenLayer, outputLayer])
model.Compile(32)
model

Model summary
  Encoder: RateEncode(nUnits=100)
  Layers:
    [0] SAMLayer(nUnits=200, gradAct=Step(a=0, b=6, g=18), teacherAct=Step(a=0, b=6, g=18))
    [1] SAMLayer(nUnits=2, gradAct=Step(a=0, b=6, g=18), teacherAct=Step(a=0, b=6, g=18))
  Decoder: MajorityDecode
  tC: 32

# 3. 学習
学習用のデータセットを利用して、SAMACTを学習させる。

## 3-1. 学習プロパティの設定
学習の際のパラメータを設定する。

In [12]:
learnProperty = LearningProperty(eta=5, iota=2, decayPeriod=1)

## 3-2. 学習の実施
フレームワークのFitメソッドを利用してモデルを学習させる。

In [13]:
fitResult = model.Fit(trainData, trainLabel, 5, learnProperty)

In [14]:
# エポックごとの学習精度
fitResult.metrics

[1.0, 1.0, 1.0, 1.0, 1.0]

## 3-3. 学習結果の保存
学習結果(学習可能パラメータ)をnpzファイルとして保存する。

In [15]:
model.Save('fitResultSample.h5')

# 4. 推論
SAMACTモデルにデータセットを推論させる。

## 4-1. テスト用データセットをまとめて推論
事前にテスト用に温存していたデータセットをSAMACTに推論させる。

In [16]:
evalResult = model.Evaluate(testData, testLabel)
evalResult.metrics

1.0

## 4-2. 1データを推論
1つのデータを推論する。(デモアプリなどで利用する想定)

In [17]:
predict = model.Predict(testData[0])
f'prediction is {predict}, golden is {testLabel[0]}'

'prediction is 0, golden is 0'